In [25]:
import pandas as pd
import ollama
from tqdm.auto import tqdm


In [20]:
df_attack = pd.read_csv('data/DATA_victims_country_corrected.csv')
# Show the column names
print(df_attack.columns)
df_attack['Victims']


Index(['Title', 'Date', 'Affiliations', 'Description', 'Response', 'Victims',
       'victims_country', 'Sponsor', 'Type', 'Category', 'Sources_1',
       'Sources_2', 'Sources_3'],
      dtype='object')


0      U.S. Department of State, U.S. Department of D...
1                               U.S. Department of State
2                             U.S. Department of Defense
3                                      Naval War College
4                            National Defense University
                             ...                        
792    Sixteen national and international sporting an...
793           South Korean organizations and individuals
794                     Taiwanese financial institutions
795               U.S. diplomats and Ugandan journalists
796                        United States, United Kingdom
Name: Victims, Length: 797, dtype: object

In [ ]:
# Define the new column names
llama_column_name = 'victim_country_llama3_1'
iso_column_name = 'victim_country_ISO'

# Get the index of the 'victims_country' column
victim_country_index = df_attack.columns.get_loc('victims_country')

# Insert the new columns if they don't exist
if llama_column_name not in df_attack.columns:
    df_attack.insert(victim_country_index + 1, llama_column_name, '')
if iso_column_name not in df_attack.columns:
    df_attack.insert(victim_country_index + 2, iso_column_name, '')


# Iterate over the DataFrame rows
with tqdm(total=len(df_attack), position=0) as pbar:
    for index, row in df_attack.iterrows():
        # Check if 'Victims' is a string
        if isinstance(row['Victims'], str):
            # Create the prompt
            str_prompt = f"Based on the following text, EXTRACT the **COUNTRY** of the text and its ISO 3166-1 alpha-3 code. Provide **ONLY** the country name and ISO code, separated by a comma. <example>When encounter: NASA. You should reply: United States, USA</example><CONSTRAINTS>Provide **ONLY** the country name and ISO code, separated by a comma. DO NOT INCLUDE ANY OTHER TEXT.</CONSTRAINTS>Text: {row['Victims']}"
            
            # Query the Llama 3.1 model
            response = ollama.chat(
                model='llama3.1',
                messages=[{'role': 'user', 'content': str_prompt}],
            )
            
            # Extract the country and ISO code from the response
            response_text = response['message']['content'].strip()
            
            str_country = ''
            str_iso = ''

            if ',' in response_text:
                parts = response_text.split(',')
                str_country = parts[0].strip()
                str_iso = parts[1].strip()
            else:
                str_country = response_text
            
            # print(f"Row {index}: {str_country}, {str_iso}")
            # if index == 10:
            #     break
            
            # Store the data in the new columns
            df_attack.at[index, llama_column_name] = str_country
            df_attack.at[index, iso_column_name] = str_iso
          
        else:
            # Handle cases where 'Victims' is not a string
            # print(f"Skipping row {index} as 'Victims' is not a string: {row['Victims']}")
            continue
        pbar.update(1)

# Display the updated DataFrame
print(df_attack[['Victims', 'victims_country', llama_column_name, iso_column_name]].head())


 97%|█████████▋| 770/797 [14:59<00:31,  1.17s/it]

                                             Victims  \
0  U.S. Department of State, U.S. Department of D...   
1                           U.S. Department of State   
2                         U.S. Department of Defense   
3                                  Naval War College   
4                        National Defense University   

                                     victims_country victim_country_llama3_1  \
0  United States; State; Defense; Department of; ...           United States   
1                               United States; State           United States   
2              United States; Defense; Department of           United States   
3                                      United States           United States   
4                            United States; National           United States   

                                  victim_country_ISO  
0  USA \n\n(There is no victim mentioned in the t...  
1                                                USA  
2                

In [29]:
df_attack[[llama_column_name, iso_column_name]].head()

,victim_country_llama3_1,victim_country_ISO
0,United States,USA \n\n(There is no victim mentioned in the t...
1,United States,USA
2,United States,USA
3,United States,USA
4,United States,USA


In [31]:
# Save the updated DataFrame to a new CSV file
df_attack.to_csv('attacks_updated.csv', index=False)

